In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any


@dataclass
class LearningEvidence:
    """
    Evidence for one concept.
    """

    concept_id: str
    score: float
    evidence: list[str] = field(
        default_factory=list
    )
    evidence_type: str = "assessment"


@dataclass
class LearningJob:
    """
    Learning evidence produced by an assessment.

    Supports both:
        1. legacy single-concept jobs
        2. multi-concept evidence
    """

    user_id: str
    project_id: str
    concept_id: str | None = None
    assessment_score: float = 0.0

    evidence: list[str] = field(
        default_factory=list
    )

    evidence_type: str = "assessment"

    concept_evidence: list[
        LearningEvidence
    ] = field(
        default_factory=list
    )

    created_at: datetime = field(
        default_factory=lambda:
            datetime.now(timezone.utc)
    )


class LearningWorker:
    """
    Updates persistent learning state after assessment evidence.

    Pipeline:

        assessment
             ↓
        concept-level evidence
             ↓
        mastery updates for affected concepts
             ↓
        recommendation refresh
    """

    def __init__(
        self,
        mastery_service=None,
        recommendation_service=None,
        database=None,
    ):

        self.mastery_service = (
            mastery_service
        )

        self.recommendation_service = (
            recommendation_service
        )

        self.database = database

    async def process(
        self,
        job: LearningJob,
    ) -> dict[str, Any]:

        if self.mastery_service is None:
            raise RuntimeError(
                "MasteryService has not been configured."
            )

        evidence_items = []

        # ----------------------------------------------------
        # NEW MULTI-CONCEPT PATH
        # ----------------------------------------------------

        if job.concept_evidence:

            evidence_items.extend(
                job.concept_evidence
            )

        # ----------------------------------------------------
        # LEGACY SINGLE-CONCEPT PATH
        # ----------------------------------------------------

        elif job.concept_id:

            evidence_items.append(
                LearningEvidence(
                    concept_id=job.concept_id,
                    score=job.assessment_score,
                    evidence=job.evidence,
                    evidence_type=job.evidence_type,
                )
            )

        else:

            raise ValueError(
                "LearningJob must contain either "
                "concept_id or concept_evidence."
            )

        # ----------------------------------------------------
        # UPDATE EACH CONCEPT
        # ----------------------------------------------------

        mastery_results = []

        for item in evidence_items:

            if not item.concept_id:
                continue

            evidence_description = None

            if item.evidence:

                evidence_description = (
                    "; ".join(
                        str(value)
                        for value in item.evidence[-3:]
                    )
                )

            mastery = (
                self.mastery_service
                .update_from_evidence(
                    user_id=job.user_id,
                    project_id=job.project_id,
                    concept_id=item.concept_id,
                    evidence_score=item.score,
                    evidence_type=item.evidence_type,
                    evidence_description=(
                        evidence_description
                    ),
                )
            )

            mastery_results.append(
                mastery
            )

        # ----------------------------------------------------
        # RECOMMENDATIONS
        # ----------------------------------------------------

        recommendations = []

        if (
            self.recommendation_service
            is not None
        ):

            try:

                recommendations = (
                    self.recommendation_service
                    .generate_for_project(
                        user_id=job.user_id,
                        project_id=job.project_id,
                    )
                )

            except Exception:
                # Mastery updates have already succeeded.
                # Recommendation failure must not erase
                # learner state.
                recommendations = []

        return {
            "user_id": job.user_id,
            "project_id": job.project_id,
            "concept_ids": [
                item.concept_id
                for item in evidence_items
                if item.concept_id
            ],
            "mastery": mastery_results,
            "recommendations": recommendations,
            "status": "completed",
        }
